In [ ]:
!sudo apt-get install zstd
# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el servidor de Ollama en segundo plano
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama)
thread.start()
time.sleep(5) # Dar tiempo a que el servidor inicie
print("Servidor de Ollama iniciado.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,111 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently

In [ ]:
!ollama pull  gemma2:9b #gemma2:9b #deepseek-r1 #llama3 #ministral-3:14b #qwen2.5:14b #mistral:7b #gemma3:12b #mistral-small3.1:24b #mistral-nemo:12b #ministral-3:14b #gemma4 #falcon3:10b  #qwen2.5:14b #ministral-3:14b c92647822/Gemini2PRO:latest #llama3.2:3b #embeddinggemma:300m #qwen2.5:3b  #qwen2.5:7b #mistral:7b  #gemma3:4b

In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
# # HISEMOTIONS 2026 — Detección de Emociones en Español Histórico
#
# ## Descripción del Reto
# - **Tarea**: Clasificación multi-label de 6 emociones en fragmentos de cartas históricas
# - **Emociones**: `anger`, `fear`, `joy`, `sadness`, `surprise`, `hope`
# - **Idioma**: Español temprano moderno (siglos XVI-XVII) ⚠️ Cambio semántico
# - **Evaluación**: Macro-F1, Precision, Recall
#
# ## Opciones de LLM Disponibles
# | Método | Configuración | Ventaja |
# |--------|---------------|---------|
# | Prompting Zero-Shot | `METHOD = "prompt_zeroshot"` | Sin entrenamiento, rápido |
# | Prompting Few-Shot | `METHOD = "prompt_fewshot"` | Mejor contexto, más preciso |
# | Fine-tuning HF | `METHOD = "hf_finetune"` | Adaptación al dominio histórico |
# | Embeddings + Classifier | `METHOD = "embedding_classifier"` | Balance velocidad/calidad |
# | Ensemble LLMs | `METHOD = "ensemble"` | Combina múltiples predicciones |

# ## Instalación de Dependencias

!pip -q install pandas numpy scikit-learn torch transformers accelerate sentence-transformers
!pip -q install huggingface_hub ollama python-dotenv imbalanced-learn

from __future__ import annotations
import json, os, warnings, zipfile, re, csv
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    pipeline,
    TrainingArguments,
    Trainer
)
from huggingface_hub import InferenceClient

# Ollama
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False

# Imbalanced learning
try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False

warnings.filterwarnings("ignore")
load_dotenv()

# ## Configuración Global

# ========== CONFIGURACIÓN PRINCIPAL ==========
METHOD = "ensemble"  # "prompt_zeroshot" | "prompt_fewshot" | "hf_finetune" | "embedding_classifier" | "ensemble"

# ========== HUGGING FACE CONFIG ==========
HF_MODEL_BASE = "FacebookAI/xlm-roberta-large"  # RoBERTa entrenado en español histórico/moderno
HF_API_MODEL = "gemma2:9b" #"gemma2:9b" #"llama3" #"qwen2.5:14b" #"mistral:7b" #"ministral-3:14b" #"gemma3:12b" #"gemma4" #"falcon3:10b" #"ministral-3:14b" #"mistral:7b" #"gemma3:4b" #"bert-base-uncased" #"mistralai/Mistral-7B-Instruct-v0.3"
HF_API_TOKEN = os.getenv("HF_TOKEN", "")

# ========== OLLAMA CONFIG ==========
OLLAMA_MODEL = "gemma2:9b" #"gemma2:9b"#"llama3" #"qwen2.5:14b" #mistral:7b" #"gemma3:12b" #"gemma4" #"falcon3:10b"#"gemma3:12b" #"ministral-3:14b" #"mistral:7b" #"gemma3:4b"  # o "llama3:8b", "mistral", "gemma2:9b"
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

# ========== EMBEDDING CONFIG ==========
EMBEDDING_MODEL = "google/embeddinggemma-300M" #"BAAI/bge-m3" #"sentence-transformers/LaBSE" #"nomic-ai/nomic-embed-text-v1" #"cardiffnlp/twitter-xlm-roberta-base-sentiment" #"bhadresh-savani/distilbert-base-uncased-emotion" #"SamLowe/roberta-base-go_emotions" #"j-hartmann/emotion-english-distilroberta-base" #"pysentimiento/robertuito-emotion-analysis" #"FacebookAI/xlm-roberta-large" #"FacebookAI/roberta-large" #"google/flan-t5-xl" #"google/flan-t5-large" #"google/flan-t5-base" #"google-bert/bert-large-uncased" #"google/mt5-base" #"google-bert/bert-base-multilingual-cased" #"google/electra-base-discriminator" #"google/gemma-2b"#"google/embeddinggemma-300M" #"sentence-transformers/LaBSE" #"nomic-ai/nomic-embed-text-v1" #"nomic-ai/nomic-embed-text-v1.5" #"allenai/longformer-base-4096" #"microsoft/deberta-v3-base" #"deepseek-ai/deepseek-coder-1.3b-base" #"intfloat/e5-mistral-7b-instruct" #"Alibaba-NLP/gte-Qwen2-1.5B-instruct" #"BAAI/bge-m3" #"paraphrase-multilingual-MiniLM-L12-v2"  # Multilingüe, funciona bien con español histórico

# ========== EMOCIONES ==========
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise", "hope"]
EMOTION_COLS = EMOTIONS  # Para alineación con CSV de salida

# ========== PARÁMETROS ==========
SEED = 42
BATCH_SIZE = 16
MAX_LENGTH = 256  # Fragmentos cortos en cartas históricas
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
THRESHOLD = 0.5  # Umbral para clasificación binaria

# ========== RUTAS ==========
TRAIN_CSV = "train.csv"
DEV_CSV = "dev.csv"
TEST_CSV = "test.csv"  # Sin labels para submission

OUTPUT_DIR = Path("")
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_CSV = "predictions.csv"
SUBMISSION_ZIP = "predictions.zip"

# ========== DEVICE ==========
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device: {device}")
print(f"✓ Method: {METHOD}")
print(f"✓ Emotions: {EMOTIONS}")

# ## Carga y Preprocesamiento de Datos Históricos

def load_hisemotions_data(filepath: str, has_labels: bool = True) -> pd.DataFrame:
    """Carga datos de HISEMOTIONS con manejo de codificación histórica"""
    # Changed separator from ',' to ','
    df = pd.read_csv(filepath, sep=',', encoding='utf-8', quoting=csv.QUOTE_NONE)

    if has_labels:
        label_cols = [col for col in EMOTIONS if col in df.columns]
        print(f"✓ Cargado: {len(df)} fragmentos | Labels: {label_cols}")
        print(f"  Distribución de emociones:\n{df[label_cols].sum().sort_values(ascending=False)}")
    else:
        print(f"✓ Cargado: {len(df)} fragmentos (sin labels)")

    return df

def preprocess_historical_text(text: str) -> str:
    """
    Preprocesamiento adaptado para español histórico (siglos XVI-XVII)
    - Preserva ortografía histórica (ç, ñ, acentos antiguos)
    - Normaliza variantes ortográficas comunes
    - Mantiene estructuras epistolares relevantes
    """
    if pd.isna(text):
        return ""

    text = str(text).strip()

    # Normalizaciones ortográficas históricas → modernas (opcional, según estrategia)
    # NOTA: Para LLMs modernos, a veces es mejor PRESERVAR la ortografía histórica
    # porque el modelo puede inferir contexto. Descomentar si se desea normalizar:
    #
    # historical_to_modern = {
    #     'ç': 'z', 'ff': 'f', 'ss': 's', 'x': 'j',  # Ejemplos simplificados
    # }
    # for old, new in historical_to_modern.items():
    #     text = text.replace(old, new)

    # Normalizar espacios múltiples y saltos de línea
    text = re.sub(r'\s+', ' ', text)

    # Preservar hasta MAX_LENGTH tokens (los fragmentos suelen ser cortos)
    tokens = text.split()
    if len(tokens) > MAX_LENGTH:
        text = ' '.join(tokens[:MAX_LENGTH])

    return text

def extract_labels(row: pd.Series) -> List[str]:
    """Extrae lista de emociones presentes (multi-label)"""
    return [emotion for emotion in EMOTIONS if row.get(emotion, 0) == 1]

# ## Estrategia 1: Prompting con LLMs (Zero-Shot / Few-Shot)

# ### Prompting Zero-Shot

class ZeroShotPromptClassifier:
    """Clasificador multi-label usando prompting zero-shot con LLM"""

    def __init__(self, method: str = "ollama", model_name: str = HF_API_MODEL):
        self.method = method
        self.model_name = model_name

        if method == "hf_api":
            print(f"☁️ Conectando a HF Inference API: {model_name}")
            self.client = InferenceClient(model=model_name, token=HF_API_TOKEN if HF_API_TOKEN else None)
        elif method == "ollama":
            print(f"🦙 Conectando a Ollama: {model_name} @ {OLLAMA_HOST}")
            self.client = ollama.Client(host=OLLAMA_HOST)
        else:
            raise ValueError(f"Método no soportado: {method}")

    def _build_prompt(self, fragment: str, examples: Optional[List[Dict]] = None) -> str:
        """Construye prompt adaptado para español histórico"""

        # Definiciones de emociones para contexto
        emotion_definitions = {
            "anger": "ira, enojo, rabia, indignación",
            "fear": "miedo, temor, espanto, preocupación",
            "joy": "alegría, placer, gozo, satisfacción",
            "sadness": "tristeza, pena, dolor, melancolía",
            "surprise": "sorpresa, asombro, extrañeza",
            "hope": "esperanza, expectativa, deseo de futuro"
        }

        definitions_str = "\n".join([f"- {emo}: {defs}" for emo, defs in emotion_definitions.items()])

        prompt = f"""Eres un experto en análisis de emociones en textos históricos en español (siglos XVI-XVII).

Tu tarea es identificar qué emociones expresa el AUTOR de este fragmento de carta histórica en el momento de escribirlo.

EMOCIONES A DETECTAR (responde con 1 si está presente, 0 si no):
{definitions_str}

INSTRUCCIONES:
1. Analiza SOLO las emociones del autor al escribir, no de terceros mencionados
2. Ignora fórmulas de cortesía epistolar (saludos, despedidas)
3. Considera el contexto histórico y semántico de la época
4. Responde SOLO con 6 números separados por comas en este orden exacto:
   anger,fear,joy,sadness,surprise,hope

FRAGMENTO HISTÓRICO:
\"{fragment}\"

Respuesta (6 números 0/1): 0 - si no se identifica, 1 - si se identifica"""

        if examples:
            examples_str = "\n\nEJEMPLOS:\n" + "\n".join([
                f'Fragmento: "{ex["text"]}"\nRespuesta: {ex["labels"]}'
                for ex in examples[:3]  # Máximo 3 ejemplos para no saturar
            ])
            prompt = prompt.replace('FRAGMENTO HISTÓRICO:', examples_str + '\n\nFRAGMENTO HISTÓRICO:')

        return prompt

    def _parse_response(self, response: str) -> List[int]:
        """Parsea respuesta del LLM a vector binario"""
        # Extraer números 0/1
        numbers = re.findall(r'[01]', response)

        if len(numbers) >= 6:
            return [int(n) for n in numbers[:6]]

        # Fallback: intentar encontrar patrón "0,1,0,0,1,0"
        pattern = re.search(r'([01]\s*,\s*){5}[01]', response)
        if pattern:
            nums = re.findall(r'[01]', pattern.group())
            if len(nums) == 6:
                return [int(n) for n in nums]

        # Default: todo en 0 si no se puede parsear
        print(f"⚠️ Response no parseable: {response[:100]}...")
        return [0] * 6

    def predict_batch(self, fragments: List[str], examples: Optional[List[Dict]] = None) -> np.ndarray:
        """Predice emociones para batch de fragmentos"""
        predictions = []

        for i, fragment in enumerate(tqdm(fragments, desc=f"{self.method} Prediction")):
            prompt = self._build_prompt(fragment, examples)

            try:
                if self.method == "hf_api":
                    response = self.client.text_generation(
                        prompt,
                        max_new_tokens=20,
                        temperature=0.1,
                        do_sample=False
                    )
                else:  # ollama
                    response = self.client.generate(
                        model=self.model_name,
                        prompt=prompt,
                        options={"temperature": 0.1, "num_predict": 20}
                    )['response']

                pred = self._parse_response(response)
                predictions.append(pred)

            except Exception as e:
                print(f"⚠️ Error en fragmento {i}: {e}")
                predictions.append([0] * 6)  # Fallback seguro

        return np.array(predictions)

# ### Prompting Few-Shot (con ejemplos del training)

class FewShotPromptClassifier(ZeroShotPromptClassifier):
    """Extiende ZeroShot con ejemplos few-shot del dataset"""

    def __init__(self, method: str = "hf_api", model_name: str = HF_API_MODEL,
                 n_examples_per_emotion: int = 2):
        super().__init__(method, model_name)
        self.n_examples = n_examples_per_emotion
        self.examples_buffer = None

    def prepare_examples(self, df_train: pd.DataFrame):
        """Prepara ejemplos balanceados por emoción para few-shot"""
        examples = []

        for emotion in EMOTIONS:
            # Filtrar fragmentos donde esta emoción = 1
            pos_samples = df_train[df_train[emotion] == 1].sample(
                n=min(self.n_examples, len(df_train[df_train[emotion] == 1])),
                random_state=SEED
            )
            for _, row in pos_samples.iterrows():
                examples.append({
                    "text": preprocess_historical_text(row["text"]),
                    "labels": ",".join(str(row[e]) for e in EMOTIONS)
                })

        # Añadir algunos ejemplos negativos (todas 0)
        neg_samples = df_train[(df_train[EMOTIONS] == 0).all(axis=1)].sample(
            n=min(2, len(df_train[(df_train[EMOTIONS] == 0).all(axis=1)])),
            random_state=SEED
        )
        for _, row in neg_samples.iterrows():
            examples.append({
                "text": preprocess_historical_text(row["text"]),
                "labels": ",".join("0" for _ in EMOTIONS)
            })

        self.examples_buffer = examples
        print(f"✓ Preparados {len(examples)} ejemplos few-shot")
        return examples

    def predict_batch(self, fragments: List[str]) -> np.ndarray:
        """Usa ejemplos preparados para few-shot"""
        return super().predict_batch(fragments, examples=self.examples_buffer)

# ## Estrategia 2: Fine-tuning de Modelo Español

class HFFineTuneClassifier:
    """Fine-tuning de modelo transformer para clasificación multi-label"""

    def __init__(self, model_name: str = HF_MODEL_BASE, num_labels: int = 6):
        print(f"🔍 Cargando modelo para fine-tuning: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            problem_type="multi_label_classification",  # Importante para multi-label
            torch_dtype=torch.float32 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
        )
        self.model_name = model_name

    def _tokenize_batch(self, texts: List[str]) -> Dict:
        return self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

    def train(self, df_train: pd.DataFrame, val_split: float = 0.1):
        """Entrena el modelo con datos históricos"""
        texts = df_train["text"].apply(preprocess_historical_text).tolist()
        labels = df_train[EMOTIONS].values.astype(float)

        # Split para validación interna
        train_texts, val_texts, train_labels, val_labels = train_test_split(
            texts, labels, test_size=val_split, random_state=SEED, stratify=labels.sum(axis=1)
        )

        # Dataset para Trainer
        class HisEmotionsDataset(torch.utils.data.Dataset):
            def __init__(self, encodings, labels):
                self.encodings = encodings
                self.labels = labels

            def __getitem__(self, idx):
                item = {k: v[idx] for k, v in self.encodings.items()}
                item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
                return item

            def __len__(self):
                return len(self.labels)

        train_enc = self._tokenize_batch(train_texts)
        val_enc = self._tokenize_batch(val_texts)

        train_dataset = HisEmotionsDataset(train_enc, train_labels)
        val_dataset = HisEmotionsDataset(val_enc, val_labels)

        # Configuración de entrenamiento adaptada para multi-label
        training_args = TrainingArguments(
            output_dir="./hisemotions_results",
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            learning_rate=LEARNING_RATE,
            weight_decay=0.01,
            warmup_ratio=0.1,
            evaluation_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            report_to="none",
            fp16=device == "cuda",
            seed=SEED
        )

        # Métrica personalizada para multi-label
        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = (torch.sigmoid(torch.tensor(logits)) > THRESHOLD).int().numpy()
            return {
                "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
                "precision": precision_score(labels, preds, average="macro", zero_division=0),
                "recall": recall_score(labels, preds, average="macro", zero_division=0)
            }

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics
        )

        print(f"🚀 Entrenando por {NUM_EPOCHS} epochs...")
        trainer.train()

        # Guardar modelo fine-tuned
        save_path = Path("./hisemotions_finetuned")
        self.model.save_pretrained(save_path)
        self.tokenizer.save_pretrained(save_path)
        print(f"✓ Modelo guardado en {save_path}")

    def predict(self, texts: List[str]) -> np.ndarray:
        """Predice emociones para nuevos fragmentos"""
        self.model.eval()
        predictions = []

        for i in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[i:i+BATCH_SIZE]
            encodings = self._tokenize_batch(batch_texts)
            encodings = {k: v.to(self.model.device) for k, v in encodings.items()}

            with torch.no_grad():
                outputs = self.model(**encodings)
                logits = outputs.logits
                # Sigmoid para multi-label + threshold
                probs = torch.sigmoid(logits)
                preds = (probs > THRESHOLD).int().cpu().numpy()
                predictions.extend(preds.tolist())

        return np.array(predictions)

# ## Estrategia 3: Embeddings + Clasificador Clásico (Recomendado para inicio)

class EmbeddingClassifier:
    """Multi-label usando embeddings multilingües + clasificadores binarios"""

    def __init__(self, embedding_model: str = EMBEDDING_MODEL, threshold: float = THRESHOLD):
        print(f"🔍 Cargando modelo de embeddings: {embedding_model}")
        from sentence_transformers import SentenceTransformer
        self.embedding_model = SentenceTransformer(embedding_model, device=device,trust_remote_code=True)
        self.threshold = threshold
        self.classifiers = {}

    def train(self, df_train: pd.DataFrame):
        """Entrena un clasificador binario por emoción"""
        texts = df_train["text"].apply(preprocess_historical_text).tolist()

        print("🚀 Generando embeddings...")
        embeddings = self.embedding_model.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # Manejar desbalance con SMOTE si está disponible
        for emotion in EMOTIONS:
            labels = df_train[emotion].values

            X, y = embeddings.copy(), labels.copy()

            if IMBLEARN_AVAILABLE and y.sum() > 0 and y.sum() < len(y):
                try:
                    # SMOTE solo si hay suficientes muestras de ambas clases
                    if y.sum() >= 5 and (1-y).sum() >= 5:
                        smote = SMOTE(random_state=SEED, k_neighbors=3)
                        X, y = smote.fit_resample(X, y)
                        print(f"  {emotion}: SMOTE aplicado ({len(y)} muestras)")
                except Exception as e:
                    print(f"  {emotion}: SMOTE omitido ({e})")

            # Clasificador por emoción (Logistic Regression funciona bien con embeddings)
            clf = LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=SEED,
                C=1.0,
                solver='lbfgs'
            )
            clf.fit(X, y)
            self.classifiers[emotion] = clf

        print(f"✓ Entrenados {len(self.classifiers)} clasificadores")

    def predict(self, texts: List[str]) -> np.ndarray:
        """Predice multi-label para nuevos fragmentos"""
        texts_proc = [preprocess_historical_text(t) for t in texts]

        embeddings = self.embedding_model.encode(
            texts_proc,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        predictions = np.zeros((len(texts), len(EMOTIONS)))

        for i, emotion in enumerate(EMOTIONS):
            if emotion in self.classifiers:
                probs = self.classifiers[emotion].predict_proba(embeddings)[:, 1]
                predictions[:, i] = (probs >= self.threshold).astype(float)

        return predictions

# ## Estrategia 4: Ensemble de Múltiples Modelos

class EnsembleClassifier:
    """Combina predicciones de múltiples estrategias para robustez"""

    def __init__(self,
                 methods: List[str] = ["embedding_classifier", "prompt_zeroshot"],
                 weights: Optional[List[float]] = None):
        self.methods = methods
        self.weights = weights or [1.0] * len(methods)
        self.classifiers = {}  # Inicializar vacío
        self.is_trained = False  # Flag para verificar si está entrenado

    def train(self, df_train: pd.DataFrame):
        """Inicializa y entrena cada clasificador del ensemble"""
        print(f"\n{'='*60}")
        print(f"🚀 Entrenando Ensemble con métodos: {self.methods}")
        print(f"{'='*60}")

        for method in self.methods:
            print(f"\n🔧 Inicializando: {method}")

            try:
                if method == "embedding_classifier":
                    clf = EmbeddingClassifier()
                    clf.train(df_train)
                    self.classifiers[method] = clf
                    print(f"   ✅ {method} entrenado correctamente")

                elif method == "hf_finetune":
                    clf = HFFineTuneClassifier()
                    clf.train(df_train)
                    self.classifiers[method] = clf
                    print(f"   ✅ {method} entrenado correctamente")

                elif method.startswith("prompt_"):
                    # Prompting no requiere entrenamiento, pero preparamos ejemplos si es few-shot
                    if method == "prompt_fewshot":
                        clf = FewShotPromptClassifier()
                        clf.prepare_examples(df_train)
                    else:  # prompt_zeroshot
                        clf = ZeroShotPromptClassifier()

                    self.classifiers[method] = clf
                    print(f"   ✅ {method} inicializado correctamente")

                else:
                    print(f"   ⚠️  Método desconocido: {method}, saltando...")

            except Exception as e:
                print(f"   ❌ Error al inicializar {method}: {str(e)}")
                print(f"   Continuando sin este método...")
                # Remover el método fallido de la lista
                if method in self.methods:
                    idx = self.methods.index(method)
                    self.methods.remove(method)
                    self.weights.pop(idx)

        # Verificar que al menos un clasificador se haya inicializado
        if not self.classifiers:
            raise ValueError("❌ No se pudo inicializar ningún clasificador del ensemble")

        self.is_trained = True
        print(f"\n{'='*60}")
        print(f"✅ Ensemble entrenado con {len(self.classifiers)} clasificadores")
        print(f"   Métodos activos: {list(self.classifiers.keys())}")
        print(f"{'='*60}\n")

    def predict(self, texts: List[str]) -> np.ndarray:
        """Combina predicciones ponderadas de todos los clasificadores"""

        # Verificar que el ensemble esté entrenado
        if not self.is_trained:
            raise RuntimeError("❌ El ensemble no ha sido entrenado. Llama a train() primero.")

        if not self.classifiers:
            raise RuntimeError("❌ No hay clasificadores disponibles en el ensemble")

        all_preds = []
        active_weights = []

        print(f"\n🔮 Prediciendo con {len(self.classifiers)} clasificadores...")
        print(f"   Clasificadores disponibles: {list(self.classifiers.keys())}")

        for method, weight in zip(self.methods, self.weights):
            # Verificar que el clasificador exista
            if method not in self.classifiers:
                print(f"   ⚠️  Saltando {method} (no disponible)")
                continue

            try:
                clf = self.classifiers[method]
                print(f"   🔄 Prediciendo con {method}...")

                if method.startswith("prompt_"):
                    # Prompting devuelve directamente predicciones
                    preds = clf.predict_batch(texts)
                else:
                    # Otros métodos usan predict() estándar
                    preds = clf.predict(texts)

                # Ponderar por peso del ensemble
                all_preds.append(preds * weight)
                active_weights.append(weight)
                print(f"   ✅ {method} completado")

            except Exception as e:
                print(f"   ❌ Error en {method}: {str(e)}")
                print(f"   Continuando sin este método...")
                continue

        # Verificar que tengamos al menos una predicción
        if not all_preds:
            raise RuntimeError("❌ Ningún clasificador pudo generar predicciones")

        # Promedio ponderado + threshold final
        ensemble_scores = np.sum(all_preds, axis=0) / sum(active_weights)
        final_preds = (ensemble_scores >= THRESHOLD).astype(float)

        print(f"   ✅ Ensemble completado\n")

        return final_preds

    def predict_proba(self, texts: List[str]) -> np.ndarray:
        """Retorna probabilidades en lugar de predicciones binarias"""

        if not self.is_trained:
            raise RuntimeError("❌ El ensemble no ha sido entrenado. Llama a train() primero.")

        all_preds = []
        active_weights = []

        for method, weight in zip(self.methods, self.weights):
            if method not in self.classifiers:
                continue

            try:
                clf = self.classifiers[method]

                if method.startswith("prompt_"):
                    preds = clf.predict_batch(texts)
                else:
                    preds = clf.predict(texts)

                all_preds.append(preds * weight)
                active_weights.append(weight)

            except Exception as e:
                print(f"   ⚠️  Error en {method}: {str(e)}")
                continue

        if not all_preds:
            raise RuntimeError("❌ Ningún clasificador pudo generar predicciones")

        # Promedio ponderado (sin threshold)
        ensemble_scores = np.sum(all_preds, axis=0) / sum(active_weights)

        return ensemble_scores

# ## Factory para Obtener Clasificador

def get_classifier(method: str = METHOD, df_train: Optional[pd.DataFrame] = None):
    """Factory para instanciar el clasificador según configuración"""

    if method == "prompt_zeroshot":
        return ZeroShotPromptClassifier(method="hf_api" if HF_API_TOKEN else "ollama")

    elif method == "prompt_fewshot":
        clf = FewShotPromptClassifier(method="hf_api" if HF_API_TOKEN else "ollama")
        if df_train is not None:
            clf.prepare_examples(df_train)
        return clf

    elif method == "hf_finetune":
        return HFFineTuneClassifier()

    elif method == "embedding_classifier":
        return EmbeddingClassifier()

    elif method == "ensemble":
        return EnsembleClassifier(methods=["embedding_classifier", "prompt_zeroshot"])

    else:
        raise ValueError(f"Método no reconocido: {method}")


# ## Funciones de Evaluación

def evaluate_multilabel(y_true: np.ndarray, y_pred: np.ndarray,
                       emotions: List[str] = EMOTIONS) -> Dict[str, int]:
    """Evalúa clasificación multi-label con métricas oficiales"""

    metrics = {}

    # Métricas globales
    for avg in ["macro", "micro", "weighted"]:
        try:
            metrics[f"f1_{avg}"] = f1_score(y_true, y_pred, average=avg, zero_division=0)
            metrics[f"precision_{avg}"] = precision_score(y_true, y_pred, average=avg, zero_division=0)
            metrics[f"recall_{avg}"] = recall_score(y_true, y_pred, average=avg, zero_division=0)
        except:
            metrics[f"f1_{avg}"] = 0.0

    # Métricas por emoción (para diagnóstico)
    per_emotion = {}
    for i, emotion in enumerate(emotions):
        try:
            f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
            per_emotion[emotion] = f1
        except:
            per_emotion[emotion] = 0.0

    metrics["per_emotion_f1"] = per_emotion

    # Reporte detallado
    print("\n" + "="*60)
    print("HISEMOTIONS - Evaluación Multi-Label")
    print("="*60)
    print(f"Macro-F1: {metrics['f1_macro']:.4f} ← Métrica principal del leaderboard")
    print(f"Micro-F1: {metrics['f1_micro']:.4f}")
    print(f"Precision: {metrics['precision_macro']:.4f}")
    print(f"Recall: {metrics['recall_macro']:.4f}")
    print("\nF1 por emoción:")
    for emo, f1 in per_emotion.items():
        bar = "█" * int(f1 * 20)
        print(f"  {emo:10s}: {f1:.3f} {bar}")

    return metrics

# ## Generación de Submission (Formato Codabench)

def generate_submission(predictions: np.ndarray, output_path: Path):
    """Genera archivo predictions.csv en formato requerido"""

    df_pred = pd.DataFrame(predictions, columns=EMOTION_COLS)

    # Guardar CSV sin índice, con tabulador como en el dataset original
    df_pred.to_csv(output_path, sep=',', index=False, quoting=csv.QUOTE_NONE)
    print(f"✓ Submission guardado: {output_path}")
    print(f"  Formato: {len(df_pred)} filas × {len(EMOTION_COLS)} columnas")

    return df_pred

def create_submission_zip(csv_path: Path, zip_path: Path):
    """Crea ZIP con predictions.csv para upload a Codabench"""

    with zipfile.ZipFile(zip_path, 'w') as zipf:
        # El CSV debe estar en la raíz del ZIP, NO en subcarpetas
        zipf.write(csv_path, arcname="predictions.csv")

    print(f"✓ ZIP creado: {zip_path}")
    print(f"  Contenido: predictions.csv ({csv_path.stat().st_size} bytes)")

def clean_dataframe_nan(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    """
    Sustituye valores NaN por valores por defecto en lugar de eliminar filas.

    Args:
        df: DataFrame a limpiar
        is_train: Si es True, también limpia y_true; si es False, limpia task_1/task_2

    Returns:
        DataFrame limpio sin NaN
    """
    df_clean = df.copy()

    # Columnas de texto → reemplazar con string vacío
    text_cols = ["anger", "fear","joy", "sadness", "surprise", "hope"]
    for col in text_cols:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(0)
            # También asegurar que sean strings
            #df_clean[col] = df_clean[col].astype(str)
    return df_clean
# ## Pipeline Principal de Ejecución

def run_pipeline(df_train: pd.DataFrame, df_dev: pd.DataFrame, df_test: pd.DataFrame,
                 method: str = METHOD) -> Tuple[np.ndarray, Dict]:
    """Ejecuta pipeline completo de entrenamiento y evaluación"""

    print("\n" + "="*60)
    print(f"HISEMOTIONS 2026 — Método: {method}")
    print("="*60)
    print("\n" + "="*60)

    # ✅ LIMPIAR NaN ANTES DE PREPROCESAR
    print(f"\n🔍 Verificando NaN antes de entrenamiento...")
    nan_counts_before = df_train.isna().sum().to_dict()
    print(f"  NaN originales: {sum(nan_counts_before.values())}")

    df_train = clean_dataframe_nan(df_train, is_train=True)
    df_dev = clean_dataframe_nan(df_dev, is_train=True)
    df_test = clean_dataframe_nan(df_test, is_train=False)

    nan_counts_after = df_train.isna().sum().to_dict()
    print(f"  NaN después de limpiar: {sum(nan_counts_after.values())}")

    # Preprocesar textos
    df_train["text_proc"] = df_train["text"].apply(preprocess_historical_text)
    df_dev["text_proc"] = df_dev["text"].apply(preprocess_historical_text)

    # Obtener clasificador
    classifier = get_classifier(method, df_train if "prompt_fewshot" in method else None)
    print(f"   ✅ Clasificador inicializado: {type(classifier).__name__}")

    # Entrenar (si aplica)
    if hasattr(classifier, 'train') and method not in ["prompt_zeroshot", "prompt_fewshot"]:
        classifier.train(df_train)
    print("Voy bien 2")
    # Predecir en desarrollo para evaluación
    print("\n🔍 Prediciendo en conjunto de desarrollo...")
    dev_texts = df_dev["text_proc"].tolist()
    predictions = classifier.predict(dev_texts) if hasattr(classifier, 'predict') else classifier.predict_batch(dev_texts)

    # Evaluar
    y_true = df_dev[EMOTIONS].values
    metrics = evaluate_multilabel(y_true, predictions)

    """Genera submission para el test set (sin labels)"""

    print(f"\n🚀 Generando submission para test set con método: {method}")
    # Si es un método que requiere entrenamiento, cargar desde checkpoint
    if method == "hf_finetune" and os.path.exists("./hisemotions_finetuned"):
        print("📦 Cargando modelo fine-tuned desde checkpoint...")
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("./hisemotions_finetuned")
        model = AutoModelForSequenceClassification.from_pretrained(
            "./hisemotions_finetuned",
            num_labels=6,
            problem_type="multi_label_classification",
            torch_dtype=torch.float16 if device == "cuda" else torch.float16
        ).to(device)
        # Reemplazar el clasificador con el cargado
        classifier.model = model
        classifier.tokenizer = tokenizer

    # Preprocesar
    df_test["text_proc"] = df_test["text"].apply(preprocess_historical_text)

    # Predecir
    test_texts = df_test["text_proc"].tolist()
    predictions = classifier.predict(test_texts) if hasattr(classifier, 'predict') else classifier.predict_batch(test_texts)

    # Generar submission
    generate_submission(predictions, SUBMISSION_CSV)
    create_submission_zip(SUBMISSION_CSV, SUBMISSION_ZIP)

    # ## Ejecución Completa
    return predictions, metrics, classifier, SUBMISSION_ZIP

# ## Ejecución Completa

if __name__ == "__main__":
    print("="*60)
    print("HISEMOTIONS 2026 — Solución con LLMs")
    print("="*60)

    # ========== CARGAR DATOS ==========
    print("\n📂 Cargando datasets...")

    # Verificar archivos
    if not os.path.exists(TRAIN_CSV):
        print(f"⚠️ No se encontró {TRAIN_CSV}")
        print("📥 Descarga los datos desde: https://github.com/albinasarymsakova/HISEMOTIONS")
    else:
        # Cargar datos con labels
        df_train = load_hisemotions_data(TRAIN_CSV, has_labels=True)
        df_dev = load_hisemotions_data(DEV_CSV, has_labels=True)
        df_test = load_hisemotions_data(TEST_CSV, has_labels=True)

        # Ejecutar pipeline de evaluación
        pred_dev, metrics, zip_path  = run_pipeline(df_train, df_dev, df_test, method=METHOD)

        # Guardar métricas
        metrics_path = OUTPUT_DIR / "metrics.json"
        with open(metrics_path, 'w', encoding='utf-8') as f:
            json.dump({
                "method": METHOD,
                "macro_f1": metrics.get("f1_macro", 0),
                "per_emotion": metrics.get("per_emotion_f1", {})
            }, f, indent=2, ensure_ascii=False)
        print(f"\n✓ Métricas guardadas: {metrics_path}")

    print(f"\n🎯 Submission lista para upload: {zip_path}")
    print("   Sube este ZIP a Codabench en 'My Submissions'")
    print("\n" + "="*60)
    print("✅ Pipeline completado")
    print("="*60)

# ## Tips Específicos para Español Histórico

# ```
# 🔹 Desafíos del español temprano moderno:
#    1. Cambio semántico: palabras con significado diferente hoy
#       Ej: "esperar" podía significar "aguardar" o "tener esperanza"
#
#    2. Ortografía variable: ç/x/j, ff/f, dobles consonantes
#       → Los LLMs modernos pueden manejar esto, pero fine-tuning ayuda
#
#    3. Estructuras epistolares: fórmulas fijas de cortesía
#       → Preprocesamiento debe identificar y marcar estas secciones
#
#    4. Expresión emocional indirecta: metáforas, referencias bíblicas
#       → Prompting con contexto histórico mejora la detección
#
# 🔹 Estrategias recomendadas:
#    1. Para baseline rápido: `embedding_classifier` con MiniLM multilingüe
#    2. Para máxima precisión: `hf_finetune` con RoBERTa español + datos históricos
#    3. Para explorar sin GPU: `prompt_zeroshot` con Mistral-7B vía API
#    4. Para robustez: `ensemble` combinando embeddings + prompting
#
# 🔹 Ajuste de threshold:
#    - El dataset está desbalanceado (hope/joy más frecuentes que surprise/fear)
#    - Considera ajustar threshold por emoción en validación:
#      thresholds = {emo: 0.4 if emo in ["hope", "joy"] else 0.6 for emo in EMOTIONS}
#
# 🔹 Augmentación de datos históricos:
#    - Back-translation a español moderno y viceversa
#    - Paráfrasis con LLM preservando ortografía histórica
#    - Synthetic examples generados con prompting controlado
# ```

# ## Comparativa Esperada de Métodos

# | Método | Macro-F1 Esperado | Velocidad | GPU | Notas |
# |--------|-----------------|-----------|-----|-------|
# | Baseline (LLM-annotated) | ~0.45-0.55 | - | - | Datos de partida |
# | `prompt_zeroshot` | ~0.50-0.60 | ⚡⚡ | No | Rápido, sin entrenamiento |
# | `prompt_fewshot` | ~0.55-0.65 | ⚡ | No | Mejora con ejemplos |
# | `embedding_classifier` | ~0.60-0.70 | ⚡⚡⚡ | Opcional | Recomendado para inicio |
# | `hf_finetune` | ~0.65-0.75 | ⚡ | Sí | Mejor adaptación al dominio |
# | `ensemble` | ~0.70-0.80 | ⚡ | Sí/No | Combina fortalezas |

# ## Notas Finales para la Competencia

# - ✅ **Formato de submission**: `predictions.csv` con tabulador, columnas en orden exacto
# - ✅ **ZIP sin carpetas**: El CSV debe estar en la raíz del `predictions.zip`
# - ✅ **Métrica principal**: **Macro-F1** para ranking en leaderboard
# - ✅ **Clases desbalanceadas**: Priorizar recall en emociones raras (surprise, fear)
# - ✅ **Contexto histórico**: Los prompts deben incluir definiciones de emociones en contexto del siglo XVI-XVII
# - ✅ **Reproducibilidad**: Incluir código y seeds en la submission final
#
# **¡Éxito en HISEMOTIONS 2026!** 🎭📜
# %%


In [ ]:
# %%
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ Memoria GPU liberada")

In [ ]:
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ Memoria GPU liberada")